# 2주차: 자연어를 구조화된 요청으로
## Structured Output

### 학습 목표
- Pydantic으로 사용자 요청을 `personal_schedule`, `group_schedule`, `todo`, `reminder`, `unknown`으로 구조화한다.
- `title`, `date`, `start_time`, `end_time`, `members`, `priority`, `reason`을 검증 가능한 payload로 만든다.
- 자유 문장 답변과 structured output payload의 차이를 설명한다.

### 핵심 개념
Structured output은 모델 답변을 앱에서 바로 쓰기 좋은 양식으로 받는 방법이다.  
Pydantic 모델을 사용하면 요청 종류와 필드를 검증 가능한 객체로 받을 수 있다.

## 환경 설정

In [ ]:
import json
import sys
sys.path.insert(0, '..')

from student_parts.week02_structure_natural_language_requests import (
    StructuredRequest,
    StructuredRequestBatch,
)
from fixed.runtime_clock import current_app_date_iso

print(f"오늘 날짜 (base_date): {current_app_date_iso()}")

## 1. 자유 문장 답변 vs Structured Output 비교

**자유 문장**: "내일 오후 3시에 철수랑 회의 잡아줘" → "네, 내일 오후 3시에 철수와 회의를 잡겠습니다!"  

**문제점**: 앱이 이 문자열에서 날짜, 시간, 참석자를 어떻게 추출할까?

In [ ]:
# Structured Output으로 구조화하면 앱이 바로 읽을 수 있는 데이터가 된다
request = StructuredRequest(
    kind="group_schedule",
    title="철수와 회의",
    date="2026-07-09",
    start_time="15:00",
    end_time=None,
    members=["철수"],
    priority=None,
    reason="사용자가 '철수랑 회의'라고 명시했으므로 group_schedule로 분류",
    original_text="내일 오후 3시에 철수랑 회의 잡아줘",
)
print("=== StructuredRequest ===")
print(json.dumps(request.model_dump(), indent=2, ensure_ascii=False))

## 2. 다섯 가지 요청 종류(kind) 구조화 비교

각 kind별로 어떤 필드가 채워지는지 확인합니다.

In [ ]:
# 1) personal_schedule: 개인 일정
ps = StructuredRequest(
    kind="personal_schedule",
    title="개인 독서 시간",
    date="2026-07-10",
    start_time="09:00",
    end_time="10:00",
    members=[],
    priority=None,
    reason="참석자 없이 혼자 하는 활동이므로 personal_schedule",
    original_text="다음 주 목요일 9시부터 10시까지 독서 시간 잡아줘",
)

# 2) group_schedule: 그룹 일정
gs = StructuredRequest(
    kind="group_schedule",
    title="팀 주간 회의",
    date="2026-07-14",
    start_time="14:00",
    end_time="15:00",
    members=["영희", "민수", "지은"],
    priority=None,
    reason="여러 멤버가 참석하는 회의이므로 group_schedule",
    original_text="다음 주 월요일 2시에 영희, 민수, 지은이랑 주간 회의",
)

# 3) todo: 할 일
td = StructuredRequest(
    kind="todo",
    title="보고서 작성",
    date="2026-07-11",
    start_time=None,
    end_time=None,
    members=[],
    priority="high",
    reason="구체적 시간 없이 마감일만 있는 작업이므로 todo",
    original_text="금요일까지 보고서 작성해야 해. 급한 거야",
)

# 4) reminder: 알림
rm = StructuredRequest(
    kind="reminder",
    title="약 먹기",
    date="2026-07-08",
    start_time="21:00",
    end_time=None,
    members=[],
    priority=None,
    reason="특정 시간에 알림만 필요한 요청이므로 reminder",
    original_text="오늘 밤 9시에 약 먹으라고 알려줘",
)

# 5) unknown: 애매한 질문
uk = StructuredRequest(
    kind="unknown",
    title=None,
    date=None,
    start_time=None,
    end_time=None,
    members=[],
    priority=None,
    reason="일정/할일/알림 중 어디에도 해당하지 않는 일반 질문",
    original_text="오늘 날씨 어때?",
)

all_requests = {
    "personal_schedule": ps,
    "group_schedule": gs,
    "todo": td,
    "reminder": rm,
    "unknown": uk,
}

for kind, req in all_requests.items():
    print(f"\n{'='*50}")
    print(f"kind: {kind}")
    print(json.dumps(req.model_dump(), indent=2, ensure_ascii=False))

## 3. kind별 필드 비교 표

In [ ]:
# kind별 필드 비교 표 출력
header = f"{'kind':20s} | {'title':12s} | {'date':12s} | {'start_time':12s} | {'end_time':12s} | {'members':18s} | {'priority':10s}"
print(header)
print("-" * len(header))

for kind, req in all_requests.items():
    d = req.model_dump()
    print(
        f"{d['kind']:20s} | "
        f"{(d['title'] or '—'):12s} | "
        f"{(d['date'] or '—'):12s} | "
        f"{(d['start_time'] or '—'):12s} | "
        f"{(d['end_time'] or '—'):12s} | "
        f"{(', '.join(d['members']) if d['members'] else '—'):18s} | "
        f"{(d['priority'] or '—'):10s}"
    )

## 4. StructuredRequestBatch로 묶기

In [ ]:
# 여러 요청도 하나의 batch로 묶을 수 있다
batch = StructuredRequestBatch(
    requests=[ps, gs, td, rm, uk],
)

print(f"base_date: {batch.base_date}")
print(f"요청 수: {len(batch.requests)}")
print()
for i, req in enumerate(batch.requests, 1):
    print(f"  [{i}] kind={req.kind}, title={req.title}")

print()
print("=== 전체 batch JSON ===")
print(json.dumps(batch.model_dump(), indent=2, ensure_ascii=False))

## 5. Pydantic 검증: 잘못된 kind는 거부됨

In [ ]:
from pydantic import ValidationError

try:
    bad = StructuredRequest(
        kind="invalid_kind",  # 허용되지 않는 kind
        original_text="테스트",
    )
except ValidationError as e:
    print("=== Pydantic 검증 실패 ===")
    print(e)
    print()
    print("→ kind는 personal_schedule, group_schedule, todo, reminder, unknown만 허용됩니다.")
    print("→ 이것이 structured output의 핵심 장점: 잘못된 값이 앱에 들어가기 전에 차단됩니다.")

## 응용 과제 정리: kind별 세부 필드 비교표

| kind | title | date | start_time | end_time | members | priority |
|------|-------|------|------------|----------|---------|----------|
| personal_schedule | ✅ 필수 | ✅ | ✅ | ✅ | ❌ 빈 list | ❌ |
| group_schedule | ✅ 필수 | ✅ | ✅ | ✅ | ✅ 필수 | ❌ |
| todo | ✅ 필수 | ⚠️ 마감일 | ❌ | ❌ | ❌ | ✅ |
| reminder | ✅ 필수 | ✅ | ✅ | ❌ | ❌ | ❌ |
| unknown | ❌ None | ❌ | ❌ | ❌ | ❌ | ❌ |

### 확인 질문 답변

1. **자유 문장 답변을 그대로 앱 데이터로 쓰면 어떤 문제가 생기는가?**
   - 날짜, 시간, 참석자를 파싱하는 별도 코드가 필요하고, 형식이 일정하지 않아 오류가 잦다.
   - "내일", "다음 주 월요일" 같은 상대 날짜를 코드로 해석해야 한다.
   - 모델이 답변 형식을 바꾸면 앱 전체가 깨질 수 있다.

2. **personal_schedule과 group_schedule은 어떤 필드에서 가장 크게 달라지는가?**
   - `members` 필드. personal_schedule은 빈 list이고, group_schedule은 참석자가 반드시 존재한다.

3. **unknown 결과는 실패인가, 아니면 안전한 분류인가?**
   - **안전한 분류**이다. 일정/할일/알림 중 어디에도 해당하지 않는 요청을 억지로 분류하면 오히려 잘못된 데이터가 생성된다.
   - unknown으로 분류하면 앱이 "이 요청은 일정 관련이 아닙니다"라고 안내할 수 있다.